# Experimento 07 — Comparação Geral de Modelos + Inferência C2DB

Notebook de análise unificada. Carrega os resultados de todos os modelos treinados
e produz comparações métricas, gráficos e análises especializadas.

## Modelos cobertos

| ID | Cenário | Checkpoint |
|----|---------|------------|
| A  | MEGNet Scratch  | `final/runs/001_megnet_finetune_c2db/model/scratch/best-*.ckpt` |
| B  | MEGNet Fine-tune | `final/runs/001_megnet_finetune_c2db/model/finetune/best-*.ckpt` |

## Estrutura

1. Carregamento dos resultados salvos (predições no test set, run 007 + run 008)
2. Tabela de métricas consolidada
3. Curvas de convergência (train/val MAE × época)
4. Scatter predito vs real por modelo
5. Viés por faixa de gap
6. Distribuição de erro absoluto (violin)
7. Triagem UWBG — inferência full database (run 005)
8. Análise de Óxidos 2D (todas as classes)
9. Verificação de cobertura de óxidos nos candidatos

**Nota:** as predições no test set vêm dos runs 007 e 008, que usaram o mesmo
split seed=42 do run 005. A triagem full-database usa `uwbg_candidates.csv` do run 005.

In [ ]:
from __future__ import annotations

import os
import re
import sys
import warnings
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.metrics import (
    mean_absolute_error, precision_recall_curve, average_precision_score,
    confusion_matrix,
)

warnings.simplefilter('ignore')

NOTEBOOK_DIR = Path(os.path.abspath('')).resolve()
_candidate_roots = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
FINAL_ROOT = next((p if p.name == 'final' else p / 'final' for p in _candidate_roots if p.name == 'final' or (p / 'final').is_dir()), None)
if FINAL_ROOT is None:
    raise RuntimeError(f'Não foi possível localizar final/ a partir de {NOTEBOOK_DIR}')
FINAL_ROOT = FINAL_ROOT.resolve()
if str(FINAL_ROOT) not in sys.path:
    sys.path.insert(0, str(FINAL_ROOT))
from common import DATA_DIR, RUNS_DIR, ensure_run_dir, required_path

RUN001 = RUNS_DIR / '001_megnet_finetune_c2db'
RUN003 = RUNS_DIR / '003_domain_gap'
RUN_DIR = ensure_run_dir('005', 'inference_screening')
FIGURES_DIR = RUN_DIR / 'figures'
OUTPUTS_DIR = RUN_DIR / 'outputs'

UWBG_THRESHOLD = 3.4

print('Caminhos configurados.')
for run_path in [RUN001, RUN003, RUN_DIR]:
    exists = '✓' if run_path.exists() else '✗ NÃO ENCONTRADO'
    print(f'  {exists}  {run_path}')

## 1. Carregamento das Predições Salvas

Os runs 007 e 008 salvaram predições no test set (seed=42, 729 amostras).
Carregamos esses CSVs diretamente — sem precisar re-executar treino ou inferência.

In [ ]:
# MEGNet Scratch (A) — run 003 (domain gap)
df_scratch = pd.read_csv(RUN003 / 'outputs/predictions_scratch.csv')

# MEGNet Fine-tune (B) — run 003 (domain gap)
df_ft = pd.read_csv(RUN003 / 'outputs/predictions_finetune.csv')

# Triagem full-database (run 001 — melhor modelo)
df_screen = pd.read_csv(RUN001 / 'outputs/uwbg_candidates.csv')

for name, df in [('Scratch (A)', df_scratch), ('Fine-tune (B)', df_ft),
                 ('Full screen (001)', df_screen)]:
    print(f'{name:22s}: {len(df):>6} amostras | colunas: {list(df.columns)}')


## 2. Métricas Consolidadas

In [ ]:
def compute_metrics(df: pd.DataFrame, name: str) -> dict:
    err     = df['pred'] - df['real']
    abs_err = err.abs()
    mae     = abs_err.mean()
    rmse    = np.sqrt((err ** 2).mean())
    bias    = err.mean()

    hse_mask  = df['fidelity'] == 1
    uwbg_mask = df['real'] > UWBG_THRESHOLD

    mae_hse   = abs_err[hse_mask].mean()  if hse_mask.sum()  > 0 else np.nan
    mae_uwbg  = abs_err[uwbg_mask].mean() if uwbg_mask.sum() > 0 else np.nan
    bias_uwbg = err[uwbg_mask].mean()     if uwbg_mask.sum() > 0 else np.nan

    faixas = [(0, 2), (2, 4), (4, 3.4), (3.4, 99)]
    mae_faixas = {}
    for lo, hi in faixas:
        m = (df['real'] >= lo) & (df['real'] < hi)
        mae_faixas[f'MAE_{lo}-{hi}'] = abs_err[m].mean() if m.sum() > 0 else np.nan

    return {
        'Modelo': name,
        'N': len(df),
        'N_UWBG': int(uwbg_mask.sum()),
        'MAE': round(mae, 4),
        'RMSE': round(rmse, 4),
        'Bias': round(bias, 4),
        'MAE_HSE': round(mae_hse, 4),
        'MAE_UWBG': round(mae_uwbg, 4),
        'Bias_UWBG': round(bias_uwbg, 4),
        **{k: round(v, 4) for k, v in mae_faixas.items()},
    }


models = [
    ('MEGNet Scratch (A)',   df_scratch),
    ('MEGNet Fine-tune (B)', df_ft),
]

metrics_list = [compute_metrics(df, name) for name, df in models]
df_metrics = pd.DataFrame(metrics_list)

display_cols = ['Modelo', 'N', 'MAE', 'RMSE', 'Bias', 'MAE_HSE', 'MAE_UWBG', 'Bias_UWBG']
print('=== Métricas no test set (729 amostras, seed=42) ===')
print(df_metrics[display_cols].to_string(index=False))

faixa_cols = ['Modelo'] + [c for c in df_metrics.columns if c.startswith('MAE_') and '-' in c]
print('\n=== MAE por faixa de gap HSE real ===')
print(df_metrics[faixa_cols].to_string(index=False))


## 3. Curvas de Convergência

In [ ]:
def load_csv_log(pattern: str) -> pd.DataFrame | None:
    files = sorted(glob.glob(pattern))
    if not files:
        return None
    return pd.read_csv(files[-1])


def extract_curve(df_log: pd.DataFrame,
                  train_col: str = 'train_MAE',
                  val_col:   str = 'val_MAE') -> pd.DataFrame:
    train  = df_log.dropna(subset=[train_col])[['epoch', train_col]].copy()
    val    = df_log.dropna(subset=[val_col])  [['epoch', val_col]].copy()
    merged = pd.merge(val, train, on='epoch', how='left')
    return merged.dropna(subset=[val_col])


log_scratch = load_csv_log(str(RUN001 / 'logs/c2db_scratch/version_*/metrics.csv'))
log_ft      = load_csv_log(str(RUN001 / 'logs/c2db_finetune/version_*/metrics.csv'))

curves = {
    'MEGNet Scratch (A)':   (log_scratch, 'steelblue'),
    'MEGNet Fine-tune (B)': (log_ft,      'darkorange'),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, (label, (log_df, color)) in zip(axes, curves.items()):
    if log_df is None:
        ax.set_title(f'{label}\n(log não encontrado)')
        continue
    curve = extract_curve(log_df)
    ax.plot(curve['epoch'], curve['val_MAE'],   color=color,      lw=2,   label='Val MAE')
    ax.plot(curve['epoch'], curve['train_MAE'], color=color, ls='--', lw=1.2, alpha=0.7,
            label='Train MAE')

    best_val = curve['val_MAE'].min()
    best_ep  = curve.loc[curve['val_MAE'].idxmin(), 'epoch']
    ax.axhline(best_val, color='gray', ls=':', lw=1)
    ax.annotate(f'Best={best_val:.4f}\nép.{int(best_ep)}',
                xy=(best_ep, best_val), xytext=(5, 10),
                textcoords='offset points', fontsize=8, color='gray')

    ax.set_xlabel('Época')
    ax.set_ylabel('MAE (eV)')
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.set_ylim(bottom=0)

plt.suptitle('Curvas de Convergência — MEGNet Scratch vs Fine-tune', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'convergence_all.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Scatter Predito vs Real

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (label, df) in zip(axes, models):
    pbe_mask = df['fidelity'] == 0
    hse_mask = df['fidelity'] == 1

    ax.scatter(df.loc[pbe_mask, 'real'], df.loc[pbe_mask, 'pred'],
               alpha=0.25, s=5, color='steelblue', label='PBE', rasterized=True)
    ax.scatter(df.loc[hse_mask, 'real'], df.loc[hse_mask, 'pred'],
               alpha=0.5,  s=8, color='darkorange', label='HSE', rasterized=True)

    lim_max = max(df['real'].max(), df['pred'].max()) + 0.5
    lim_min = max(0, min(df['real'].min(), df['pred'].min()) - 0.3)
    ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', lw=1, alpha=0.7)
    ax.axvline(UWBG_THRESHOLD, color='red', ls=':', lw=0.9, alpha=0.6)
    ax.axhline(UWBG_THRESHOLD, color='red', ls=':', lw=0.9, alpha=0.6)

    row = df_metrics[df_metrics['Modelo'] == label].iloc[0]
    ax.set_title(f'{label}\nMAE={row["MAE"]:.3f} eV  |  MAE_HSE={row["MAE_HSE"]:.3f} eV')
    ax.set_xlabel('Gap real (eV)')
    ax.set_ylabel('Gap predito (eV)')
    ax.legend(fontsize=8, markerscale=2)
    ax.set_xlim(lim_min, lim_max)
    ax.set_ylim(lim_min, lim_max)

plt.suptitle('Predito vs Real — Todos os Modelos (Test Set)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'scatter_all.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Viés por Faixa de Gap

In [ ]:
fine_edges  = [0, 1, 2, 3, 4, 5, 6, 7, 20]
fine_labels = ['0–1', '1–2', '2–3', '3–4', '4–5', '5–6', '6–7', '7+']
colors_model = {
    'MEGNet Scratch (A)':   'steelblue',
    'MEGNet Fine-tune (B)': 'darkorange',
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
x = np.arange(len(fine_labels))
width = 0.35
for i, (label, df) in enumerate(models):
    df = df.copy()
    df['bin'] = pd.cut(df['real'], bins=fine_edges, labels=fine_labels, right=False)
    mae_bins = df.groupby('bin', observed=True)['abs_error'].mean()
    offset = (i - 0.5) * width
    ax.bar(x + offset, mae_bins.reindex(fine_labels).fillna(0),
           width=width, label=label, color=colors_model[label], alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(fine_labels, rotation=30, ha='right')
ax.set_xlabel('Faixa de gap HSE real (eV)')
ax.set_ylabel('MAE (eV)')
ax.set_title('MAE por faixa de gap')
ax.legend(fontsize=8)
ax.axvline(3.4 - 0.5, color='red', ls=':', lw=1, alpha=0.5)

ax2 = axes[1]
for i, (label, df) in enumerate(models):
    df = df.copy()
    df['bin']  = pd.cut(df['real'], bins=fine_edges, labels=fine_labels, right=False)
    df['bias'] = df['pred'] - df['real']
    bias_bins  = df.groupby('bin', observed=True)['bias'].mean()
    offset = (i - 0.5) * width
    ax2.bar(x + offset, bias_bins.reindex(fine_labels).fillna(0),
            width=width, label=label, color=colors_model[label], alpha=0.8)

ax2.axhline(0, color='black', lw=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(fine_labels, rotation=30, ha='right')
ax2.set_xlabel('Faixa de gap HSE real (eV)')
ax2.set_ylabel('Viés médio (eV)  [+ = superestimação]')
ax2.set_title('Viés médio por faixa')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mae_bias_per_range.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Distribuição de Erro Absoluto por Modelo (Violin)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

coarse_labels = ['0–2 eV', '2–3.4 eV', '3.4–4 eV', '>4 eV']
coarse_edges  = [0, 2, 3.4, 4, 99]

# ── Painel 1: violins por modelo (global) ────────────────────────
ax = axes[0]
data_violin = [df['abs_error'].values for _, df in models]
labels_violin = [n.split(' (')[0] for n, _ in models]  # nome curto
vp = ax.violinplot(data_violin, positions=range(len(models)),
                   showmedians=True, showextrema=False)
for body, color in zip(vp['bodies'], colors_model.values()):
    body.set_facecolor(color)
    body.set_alpha(0.6)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(labels_violin)
ax.set_ylabel('Erro absoluto (eV)')
ax.set_title('Distribuição do erro — todos os modelos')

# ── Painel 2: MAE por faixa grande (barras agrupadas) ────────────
ax2 = axes[1]
x   = np.arange(len(coarse_labels))
width = 0.25
for i, (label, df) in enumerate(models):
    df = df.copy()
    df['cbin'] = pd.cut(df['real'], bins=coarse_edges, labels=coarse_labels, right=False)
    mae_c = df.groupby('cbin', observed=True)['abs_error'].mean()
    offset = (i - 1) * width
    bars = ax2.bar(x + offset, mae_c.reindex(coarse_labels).fillna(0),
                   width=width, label=label, color=colors_model[label], alpha=0.8)

ax2.set_xticks(x)
ax2.set_xticklabels(coarse_labels, rotation=15, ha='right')
ax2.set_ylabel('MAE (eV)')
ax2.set_title('MAE por faixa — comparação direta')
ax2.legend(fontsize=8)

# Linha de referência UWBG
ax2.axvline(2.5, color='red', ls=':', lw=1, alpha=0.4)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'error_violin_range.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Curva Precision-Recall — Classificação UWBG

Usando predições HSE (fidelidade=1) de cada modelo, avaliamos a capacidade de
classificar corretamente materiais UWBG (gap HSE real > 3.4 eV).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for label, df in models:
    hse = df[df['fidelity'] == 1].copy()
    y_true = (hse['real'] > UWBG_THRESHOLD).astype(int)
    y_score = hse['pred']

    if y_true.sum() == 0:
        print(f'[!] {label}: sem amostras UWBG no subset HSE — P-R omitida')
        continue

    prec, rec, thresh = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)

    color = colors_model[label]
    ax.plot(rec, prec, color=color, lw=2, label=f'{label} (AP={ap:.3f})')

    # Ponto no limiar 3.4 eV
    thresh_idx = np.searchsorted(thresh, UWBG_THRESHOLD)
    if thresh_idx < len(prec) - 1:
        ax.scatter(rec[thresh_idx], prec[thresh_idx], color=color, s=80, zorder=5)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'Curva P-R — Classificação UWBG (gap HSE > {UWBG_THRESHOLD} eV)')
ax.legend(fontsize=9)
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'precision_recall_uwbg.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Triagem UWBG — Full Database (Run 005)

Resultado da inferência sobre todos os materiais do C2DB (modelo MEGNet Fine-tune, run 005).

In [ ]:
# Adiciona colunas derivadas
df_screen = df_screen.copy()

# Corrija nomes de coluna se necessário (compatibilidade com run 005)
if 'gap_hse_true' not in df_screen.columns and 'gap_hse' in df_screen.columns:
    df_screen.rename(columns={'gap_hse': 'gap_hse_true'}, inplace=True)

df_uwbg = df_screen[df_screen['uwbg'] == True].copy()
df_no_hse = df_uwbg[df_uwbg['gap_hse_true'].isna()].copy()
df_has_hse = df_uwbg[df_uwbg['gap_hse_true'].notna()].copy()

print(f'Total materiais triados (run 005)  : {len(df_screen):>6}')
print(f'  → com predição UWBG (> 3.4 eV)  : {len(df_uwbg):>6}')
print(f'  → com HSE confirmado             : {len(df_has_hse):>6}')
print(f'  → sem HSE (novos candidatos DFT) : {len(df_no_hse):>6}')

# Distribuição de gap predito nos candidatos UWBG
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(df_uwbg['gap_hse_pred'], bins=40, color='seagreen', alpha=0.8, edgecolor='white', lw=0.5)
ax.axvline(UWBG_THRESHOLD, color='red', ls='--', lw=1.5, label=f'UWBG {UWBG_THRESHOLD} eV')
ax.set_xlabel('HSE predito (eV)')
ax.set_ylabel('Contagem')
ax.set_title(f'Distribuição dos candidatos UWBG (n={len(df_uwbg)})')
ax.legend()

ax2 = axes[1]
ax2.scatter(df_screen['gap_pbe'], df_screen['gap_hse_pred'],
            s=4, alpha=0.2, color='steelblue', label='Não-UWBG', rasterized=True)
ax2.scatter(df_uwbg['gap_pbe'], df_uwbg['gap_hse_pred'],
            s=12, alpha=0.7, color='darkorange', label='UWBG pred', rasterized=True)
ax2.axhline(UWBG_THRESHOLD, color='red', ls='--', lw=1)
ax2.set_xlabel('Gap PBE (eV)')
ax2.set_ylabel('Gap HSE pred (eV)')
ax2.set_title('PBE vs HSE predito — todos os materiais C2DB')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'uwbg_screening_all.png', dpi=150, bbox_inches='tight')
plt.show()

# Top 20 candidatos sem HSE calculado (novos para DFT)
print(f'\n=== Top 20 novos candidatos (sem HSE calculado) ===')
top20 = df_no_hse.sort_values('gap_hse_pred', ascending=False).head(20)
print(top20[['uid', 'formula', 'gap_pbe', 'gap_hse_pred']].to_string(index=False))

## 9. Análise de Óxidos 2D

Óxidos apresentam correção PBE→HSE sistematicamente maior (~+1.5 a +3.0 eV) vs média do dataset (~+0.7 eV).

Esta seção analisa:
- Todos os óxidos no dataset de triagem (fórmula contém O)
- Subgrupo XO₂ (estequiometria simples: um metal + dois oxigênios)
- Viés sistemático: distribuição de Δ = HSE − PBE para óxidos vs restante
- Candidatos UWBG óxidos identificados pelo modelo
- Verificação se todos os óxidos conhecidos aparecem nos candidatos

In [ ]:
# ── Funções de classificação de óxidos ──────────────────────────
def has_oxygen(formula: str) -> bool:
    """Detecta óxidos: 'O' não seguido de 's' (Os=Osmium) ou 'g' (Og=Oganesson)."""
    return bool(re.search(r'O(?![sg])', str(formula)))

def is_xo2(formula: str) -> bool:
    """XO₂: exatamente um elemento + dois oxigênios (ex: TiO2, GeO2, SnO2)."""
    return bool(re.fullmatch(r'[A-Z][a-z]?O2', str(formula)))


# Aplica ao dataset de triagem completo
df_screen = df_screen.copy()
df_screen['is_oxide'] = df_screen['formula'].apply(has_oxygen)
df_screen['is_xo2']   = df_screen['formula'].apply(is_xo2)

df_ox  = df_screen[df_screen['is_oxide']]
df_xo2 = df_screen[df_screen['is_xo2']]
df_ox_uwbg = df_screen[df_screen['is_oxide'] & (df_screen['uwbg'] == True)]

print(f'Total no dataset de triagem : {len(df_screen):>5}')
print(f'Óxidos (contém O)           : {len(df_ox):>5}  ({len(df_ox)/len(df_screen)*100:.1f}%)')
print(f'  → UWBG predito            : {len(df_ox_uwbg):>5}')
print(f'XO₂ (estequiometria simples): {len(df_xo2):>5}')
print(f'  → UWBG predito            : {df_xo2["uwbg"].sum():>5}')

In [ ]:
# Tabela de candidatos UWBG óxidos
from IPython.display import display

df_ox_uwbg_sorted = df_ox_uwbg.sort_values('gap_hse_pred', ascending=False).copy()
df_ox_uwbg_sorted['subgrupo'] = df_ox_uwbg_sorted['is_xo2'].map({True: 'XO₂', False: 'outro óxido'})

display_cols = ['formula', 'subgrupo', 'gap_pbe', 'gap_hse_true', 'gap_hse_pred']
rename = {
    'formula': 'Fórmula', 'subgrupo': 'Subgrupo',
    'gap_pbe': 'PBE (eV)', 'gap_hse_true': 'HSE real (eV)', 'gap_hse_pred': 'HSE pred (eV)',
}

df_show = df_ox_uwbg_sorted[display_cols].rename(columns=rename)
styled = (
    df_show.style
    .format({'PBE (eV)': '{:.3f}', 'HSE real (eV)': '{:.3f}', 'HSE pred (eV)': '{:.3f}'},
            na_rep='—')
    .background_gradient(subset=['HSE pred (eV)'], cmap='YlOrRd')
    .set_caption(f'Candidatos UWBG óxidos — {len(df_ox_uwbg_sorted)} materiais (HSE pred > 3.4 eV)')
)
display(styled)

In [ ]:
# ── Figura 1: Scatter PBE vs HSE_pred — óxidos destacados ────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
non_ox = df_screen[~df_screen['is_oxide']]
ax.scatter(non_ox['gap_pbe'], non_ox['gap_hse_pred'],
           s=5, alpha=0.2, color='steelblue', label='Não-óxidos', rasterized=True)
ax.scatter(df_ox['gap_pbe'], df_ox['gap_hse_pred'],
           s=18, alpha=0.7, color='tomato', label='Óxidos', rasterized=True)
ax.scatter(df_xo2['gap_pbe'], df_xo2['gap_hse_pred'],
           s=40, alpha=0.9, color='darkred', marker='*', label='XO₂', zorder=5)
ax.axhline(UWBG_THRESHOLD, color='gray', ls='--', lw=1, label='UWBG 3.4 eV')
ax.set_xlabel('Gap PBE (eV)')
ax.set_ylabel('Gap HSE pred (eV)')
ax.set_title('PBE vs HSE predito — óxidos em destaque')
ax.legend(fontsize=8)

# ── Figura 2: distribuição Δ = HSE_real - PBE ────────────────────
ax2 = axes[1]
df_with_hse = df_screen.dropna(subset=['gap_hse_true']).copy()
df_with_hse['delta'] = df_with_hse['gap_hse_true'] - df_with_hse['gap_pbe']

ox_delta    = df_with_hse[df_with_hse['is_oxide']]['delta']
nonox_delta = df_with_hse[~df_with_hse['is_oxide']]['delta']

bins = np.linspace(-2, 6, 60)
ax2.hist(nonox_delta, bins=bins, alpha=0.55, color='steelblue',
         label=f'Não-óxidos (n={len(nonox_delta)}, μ={nonox_delta.mean():.2f} eV)')
ax2.hist(ox_delta, bins=bins, alpha=0.7, color='tomato',
         label=f'Óxidos (n={len(ox_delta)}, μ={ox_delta.mean():.2f} eV)')
ax2.axvline(nonox_delta.mean(), color='steelblue', lw=2, ls='--')
ax2.axvline(ox_delta.mean(), color='darkred', lw=2, ls='--')
ax2.set_xlabel('Δ = HSE − PBE (eV)')
ax2.set_ylabel('Contagem')
ax2.set_title('Distribuição da correção HSE − PBE (onde HSE real existe)')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'oxide_scatter_delta.png', dpi=150, bbox_inches='tight')
plt.show()

# Estatísticas por subgrupo
print('\n=== Correção PBE → HSE por subgrupo (onde HSE real existe) ===')
for group_label, mask in [
    ('Não-óxidos', ~df_with_hse['is_oxide']),
    ('Óxidos',      df_with_hse['is_oxide']),
    ('XO₂',         df_with_hse['is_xo2']),
]:
    sub = df_with_hse[mask]['delta']
    if len(sub) == 0:
        continue
    print(f'  {group_label:<12}: n={len(sub):>4}  μ={sub.mean():.3f} eV  '
          f'σ={sub.std():.3f} eV  min={sub.min():.2f}  max={sub.max():.2f}')

## 10. Verificação de Cobertura de Óxidos nos Candidatos

Verifica se óxidos com HSE real > 3.4 eV (conhecidos no C2DB) estão sendo identificados pelo modelo.
Permite detectar falsos negativos sistemáticos em óxidos.

In [ ]:
# Óxidos com HSE real confirmado (no test set ou full dataset)
df_ox_real = df_screen[df_screen['is_oxide'] & df_screen['gap_hse_true'].notna()].copy()
df_ox_real_uwbg = df_ox_real[df_ox_real['gap_hse_true'] > UWBG_THRESHOLD].copy()

print(f'Óxidos com HSE real no dataset      : {len(df_ox_real)}')
print(f'  → HSE real > {UWBG_THRESHOLD} eV (UWBG confirmado): {len(df_ox_real_uwbg)}')
print()

# Quantos desses foram detectados pelo modelo?
detected = df_ox_real_uwbg[df_ox_real_uwbg['uwbg'] == True]
missed   = df_ox_real_uwbg[df_ox_real_uwbg['uwbg'] != True]

recall_ox = len(detected) / len(df_ox_real_uwbg) if len(df_ox_real_uwbg) > 0 else 0
print(f'Detectados pelo modelo (UWBG pred)  : {len(detected)}  (recall = {recall_ox:.1%})')
print(f'Não detectados (falsos negativos)   : {len(missed)}')

if len(missed) > 0:
    print('\n=== Óxidos UWBG reais NÃO detectados pelo modelo ===')
    print(f'{"Fórmula":15} {"HSE real (eV)":>15} {"HSE pred (eV)":>15} {"PBE (eV)":>10}')
    print('-' * 60)
    for _, row in missed.sort_values('gap_hse_true', ascending=False).iterrows():
        print(f'  {row["formula"]:13}  {row["gap_hse_true"]:>13.3f}  '
              f'{row["gap_hse_pred"]:>13.3f}  {row["gap_pbe"]:>8.3f}')
else:
    print('\n✓ Todos os óxidos UWBG confirmados foram detectados pelo modelo.')

In [ ]:
# Figura: HSE real vs HSE pred para óxidos — identificação de falsos negativos
fig, ax = plt.subplots(figsize=(8, 6))

# Óxidos com HSE real
ax.scatter(df_ox_real['gap_hse_true'], df_ox_real['gap_hse_pred'],
           s=20, alpha=0.5, color='steelblue', label='Óxidos (geral)', rasterized=True)

# Detectados
ax.scatter(detected['gap_hse_true'], detected['gap_hse_pred'],
           s=50, alpha=0.85, color='seagreen', marker='^', label=f'UWBG real + detectado (n={len(detected)})')

# Falsos negativos
if len(missed) > 0:
    ax.scatter(missed['gap_hse_true'], missed['gap_hse_pred'],
               s=80, alpha=0.9, color='tomato', marker='x', lw=2,
               label=f'UWBG real + NÃO detectado (n={len(missed)})')
    for _, row in missed.iterrows():
        ax.annotate(row['formula'], (row['gap_hse_true'], row['gap_hse_pred']),
                    fontsize=7, xytext=(4, 4), textcoords='offset points', color='darkred')

# Diagonal
lim_max = max(df_ox_real['gap_hse_true'].max(), df_ox_real['gap_hse_pred'].max()) + 0.5
ax.plot([0, lim_max], [0, lim_max], 'k--', lw=1, alpha=0.5)
ax.axvline(UWBG_THRESHOLD, color='gray', ls=':', lw=1)
ax.axhline(UWBG_THRESHOLD, color='gray', ls=':', lw=1, label=f'Limiar UWBG {UWBG_THRESHOLD} eV')

ax.set_xlabel('HSE real (eV)')
ax.set_ylabel('HSE predito (eV)')
ax.set_title('Cobertura de óxidos — HSE real vs predito\n(modelo MEGNet Fine-tune, run 005)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'oxide_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Erro do Modelo nos Óxidos — Comparação por Modelo

Usando o test set (onde temos predições de todos os modelos e HSE real), avalia o
desempenho específico em óxidos vs não-óxidos para cada modelo.

In [ ]:
# Identifica óxidos no test set via uid_base ou formula
# df_screen tem formula; precisamos mapear uid_base → is_oxide
uid_to_oxide = df_screen.set_index('uid')['is_oxide'].to_dict()
uid_to_formula = df_screen.set_index('uid')['formula'].to_dict()

print('=== Erro por subgrupo (óxidos vs não-óxidos) no test set ===')
print(f'  {"Modelo":<24} {"Subgrupo":<14} {"N":>5} {"MAE (eV)":>10} {"Bias (eV)":>10}')
print('  ' + '-' * 70)

for label, df in models:
    # Mapeia is_oxide via uid_base
    df = df.copy()
    df['is_oxide'] = df['uid_base'].map(uid_to_oxide).fillna(False)

    # Apenas HSE (fidelidade 1, onde a comparação com UWBG faz sentido)
    df_hse = df[df['fidelity'] == 1].copy()

    for subgroup, mask in [('Não-óxidos', ~df_hse['is_oxide']), ('Óxidos', df_hse['is_oxide'])]:
        sub = df_hse[mask]
        if len(sub) == 0:
            continue
        mae  = sub['abs_error'].mean()
        bias = sub['bias'].mean()
        print(f'  {label:<24} {subgroup:<14} {len(sub):>5} {mae:>10.3f} {bias:>+10.3f}')

In [ ]:
# Figura: scatter HSE real vs pred por modelo — óxidos destacados
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (label, df) in zip(axes, models):
    df = df.copy()
    df['is_oxide'] = df['uid_base'].map(uid_to_oxide).fillna(False)
    hse = df[df['fidelity'] == 1]

    non_ox = hse[~hse['is_oxide']]
    ox     = hse[hse['is_oxide']]

    ax.scatter(non_ox['real'], non_ox['pred'],
               s=6, alpha=0.3, color='steelblue', label='Não-óxidos', rasterized=True)
    ax.scatter(ox['real'], ox['pred'],
               s=25, alpha=0.8, color='tomato', label='Óxidos', rasterized=True, zorder=4)

    lim = max(hse['real'].max(), hse['pred'].max()) + 0.5
    ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.6)
    ax.axvline(UWBG_THRESHOLD, color='gray', ls=':', lw=0.8)
    ax.axhline(UWBG_THRESHOLD, color='gray', ls=':', lw=0.8)

    mae_ox    = ox['abs_error'].mean()    if len(ox)     > 0 else np.nan
    mae_nonox = non_ox['abs_error'].mean() if len(non_ox) > 0 else np.nan
    ax.set_title(f'{label}\nMAE óxidos={mae_ox:.3f}  não-óxidos={mae_nonox:.3f} eV')
    ax.set_xlabel('HSE real (eV)')
    ax.set_ylabel('HSE predito (eV)')
    ax.legend(fontsize=8)

plt.suptitle('HSE real vs predito — óxidos destacados (test set)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'oxide_scatter_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Resumo Executivo

In [ ]:
from IPython.display import display

summary_cols = ['Modelo', 'MAE', 'RMSE', 'Bias', 'MAE_HSE', 'MAE_UWBG', 'Bias_UWBG']
df_summary = df_metrics[summary_cols].copy()

# Destaca o melhor em cada coluna numérica
styled_summary = (
    df_summary.style
    .format({c: '{:.4f}' for c in summary_cols if c != 'Modelo'})
    .highlight_min(subset=['MAE', 'RMSE', 'MAE_HSE', 'MAE_UWBG'], color='lightgreen')
    .set_caption('Resumo — métricas no test set (729 amostras, seed=42)')
)
display(styled_summary)

# Vencedor por métrica
print('\n=== Melhor modelo por métrica ===')
for col in ['MAE', 'RMSE', 'MAE_HSE', 'MAE_UWBG']:
    best_idx = df_metrics[col].idxmin()
    best_val = df_metrics.loc[best_idx, col]
    best_name = df_metrics.loc[best_idx, 'Modelo']
    print(f'  {col:<12}: {best_name}  ({best_val:.4f} eV)')

## 13. Tabela Interativa — Todos os Materiais C2DB

Tabela pesquisável com **todos os materiais do C2DB** (com e sem HSE calculado).
Predições HSE (fidelidade=1) são geradas para os dois modelos MEGNet.

**Primeira execução**: constrói os grafos e roda inferência (~5–10 min com GPU).
Resultados são salvos em cache — execuções seguintes carregam o CSV em segundos.

| Coluna | Descrição |
|--------|-----------|
| Fórmula | Fórmula química |
| Categoria | haleto / óxido / sulfeto / seleneto / telureto / oxihaleto / nitreto … |
| PBE | Gap PBE real (eV) |
| HSE_real | Gap HSE calculado (eV) — `—` onde não existe |
| Pred_FT | Predição MEGNet Fine-tune (eV) |
| Pred_SC | Predição MEGNet Scratch (eV) |
| UWBG_FT | Fine-tune prediz gap ≥ 3,4 eV |


In [ ]:
# ── Configuração ─────────────────────────────────────────────────
RERUN_INFERENCE = False
EHULL_MAX = 0.2
CUTOFF_MEGNET = 5.0
HSE_FID = 1
RUN001_INF = RUN001
CACHE_DIR = RUN_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_CSV = CACHE_DIR / 'all_materials_predictions.csv'

print(f'RUN001 : {RUN001_INF}')
print(f'Cache  : {CACHE_CSV}')
print(f'RERUN_INFERENCE: {RERUN_INFERENCE}')
print(f'Cache existe: {CACHE_CSV.exists()}')

In [ ]:
import ase.db
from pymatgen.io.ase import AseAtomsAdaptor

adaptor = AseAtomsAdaptor()
C2DB_PATH = required_path(DATA_DIR / 'raw' / 'c2db.db', 'C2DB')

if RERUN_INFERENCE or not CACHE_CSV.exists():
    db = ase.db.connect(str(C2DB_PATH))
    structures_all, meta_all = {}, {}
    skipped = 0

    for row in db.select():
        uid = row.get('uid') or str(row.id)
        ehull = row.get('ehull')
        if ehull is not None and ehull > EHULL_MAX:
            skipped += 1
            continue
        g_pbe = row.get('gap')
        g_hse = row.get('gap_hse')
        dyn   = row.get('dyn_stab')
        try:
            atoms  = row.toatoms()
            struct = adaptor.get_structure(atoms)
        except Exception:
            skipped += 1
            continue
        structures_all[uid] = struct
        meta_all[uid] = {
            'formula' : struct.formula,
            'n_atoms' : len(struct),
            'gap_pbe' : g_pbe,
            'gap_hse_true': g_hse,
            'ehull'   : ehull,
            'dyn_stab': dyn,
        }

    print(f'Materiais carregados : {len(structures_all)}')
    print(f'  com PBE            : {sum(v["gap_pbe"] is not None for v in meta_all.values())}')
    print(f'  com HSE            : {sum(v["gap_hse_true"] is not None for v in meta_all.values())}')
    print(f'  pulados (ehull>{EHULL_MAX}): {skipped}')
else:
    print('Cache encontrado — pulando carregamento do C2DB.')

In [ ]:
from tqdm.auto import tqdm

import torch
from matgl.config import DEFAULT_ELEMENTS
from matgl.models import MEGNet
from matgl.utils.training import ModelLightningModule

if RERUN_INFERENCE or not CACHE_CSV.exists():
    element_types = DEFAULT_ELEMENTS

    def build_megnet_c2db(element_types=element_types,
                          cutoff=CUTOFF_MEGNET) -> MEGNet:
        return MEGNet(
            element_types=element_types,
            cutoff=cutoff,
            is_intensive=True,
            dim_state_embedding=64,
            ntypes_state=2,
            readout_type='set2set',
            include_states=True,
        )

    # Localiza checkpoints em final/runs/001
    ckpt_ft = sorted((RUN001_INF / 'model/finetune').glob('best-*.ckpt'))[-1]
    ckpt_sc = sorted((RUN001_INF / 'model/scratch').glob('best-*.ckpt'))[-1]
    print(f'FT  ckpt: {ckpt_ft.name}')
    print(f'SC  ckpt: {ckpt_sc.name}')

    model_ft = build_megnet_c2db()
    lit_ft   = ModelLightningModule.load_from_checkpoint(str(ckpt_ft), model=model_ft, map_location='cpu')
    lit_ft.model.eval()

    model_sc = build_megnet_c2db()
    lit_sc   = ModelLightningModule.load_from_checkpoint(str(ckpt_sc), model=model_sc, map_location='cpu')
    lit_sc.model.eval()

    state_hse = torch.tensor([HSE_FID], dtype=torch.long)

    uids_list   = list(structures_all.keys())
    preds_ft    = {}
    preds_sc    = {}

    print(f'Rodando inferência MEGNet para {len(uids_list)} materiais...')
    failed = 0
    with torch.no_grad():
        for uid in tqdm(uids_list, desc='MEGNet inference'):
            struct = structures_all[uid]
            try:
                preds_ft[uid] = lit_ft.model.predict_structure(struct, state_attr=state_hse).item()
                preds_sc[uid] = lit_sc.model.predict_structure(struct, state_attr=state_hse).item()
            except Exception:
                preds_ft[uid] = float('nan')
                preds_sc[uid] = float('nan')
                failed += 1

    print(f'MEGNet concluído. Falhas: {failed}')


In [ ]:
# ALIGNN foi descartado do pipeline final (convergência instável, MAE 82% pior).
# Esta célula existe apenas como marcador estrutural.
if RERUN_INFERENCE or not CACHE_CSV.exists():
    pass  # sem ALIGNN nesta versão final


In [ ]:
import re

def classify_material(formula: str) -> str:
    f = str(formula)
    has_F  = bool(re.search(r'F(?![er])',        f))
    has_Cl = bool(re.search(r'Cl',               f))
    has_Br = bool(re.search(r'Br',               f))
    has_I  = bool(re.search(r'I(?![nr])',         f))
    is_halide    = has_F or has_Cl or has_Br or has_I
    is_nitride   = bool(re.search(r'N(?![abdeionp])', f))
    is_oxide     = bool(re.search(r'O(?![sg])',        f))
    is_sulfide   = bool(re.search(r'S(?![bceimnrb])', f))
    is_selenide  = bool(re.search(r'Se',               f))
    is_telluride = bool(re.search(r'Te',               f))
    is_phosphide = bool(re.search(r'P(?![bdmort])',    f))
    is_carbide   = bool(re.search(r'C(?![adeiorsu])',  f))
    is_boride    = bool(re.search(r'B(?![aeirn])',     f)) and not has_Br
    is_hydride   = bool(re.search(r'H(?![efg])',       f))
    if is_oxide and is_halide:    return 'oxihaleto'
    if is_oxide and is_nitride:   return 'oxinitreto'
    if is_sulfide and is_halide:  return 'tiossal'
    if is_halide:    return 'haleto'
    if is_nitride:   return 'nitreto'
    if is_oxide:     return 'óxido'
    if is_telluride: return 'telureto'
    if is_selenide:  return 'seleneto'
    if is_sulfide:   return 'sulfeto'
    if is_phosphide: return 'fosfeto'
    if is_carbide:   return 'carbeto'
    if is_boride:    return 'boreto'
    if is_hydride:   return 'hidreto'
    return 'outro'

if RERUN_INFERENCE or not CACHE_CSV.exists():
    # Monta DataFrame completo
    # Indexa predições ALIGNN pelo uid para lookup rápido
    rows = []
    for uid in uids_list:
        m = meta_all[uid]
        pred_ft = preds_ft.get(uid, float('nan'))
        pred_sc = preds_sc.get(uid, float('nan'))
        rows.append({
            'uid'        : uid,
            'formula'    : m['formula'],
            'n_atoms'    : m['n_atoms'],
            'ehull'      : m['ehull'],
            'dyn_stab'   : m['dyn_stab'],
            'gap_pbe'    : m['gap_pbe'],
            'gap_hse_true': m['gap_hse_true'],
            'Pred_FT'    : pred_ft,
            'Pred_SC'    : pred_sc,
        })
    df_all = pd.DataFrame(rows)
    df_all['categoria'] = df_all['formula'].apply(classify_material)
    df_all['UWBG_FT'] = df_all['Pred_FT'] > UWBG_THRESHOLD
    df_all.to_csv(CACHE_CSV, index=False)
    print(f'Cache salvo: {CACHE_CSV}  ({len(df_all)} materiais)')
else:
    df_all = pd.read_csv(CACHE_CSV)
    # Recalcula UWBG_FT e categoria caso o CSV já esteja salvo
    if 'categoria' not in df_all.columns:
        df_all['categoria'] = df_all['formula'].apply(classify_material)
    if 'UWBG_FT' not in df_all.columns:
        df_all['UWBG_FT'] = df_all['Pred_FT'] > UWBG_THRESHOLD
    print(f'Cache carregado: {len(df_all)} materiais')

# Resumo
print(f'  Total                  : {len(df_all)}')
print(f'  Com HSE real           : {df_all["gap_hse_true"].notna().sum()}')
print(f'  Com Pred_FT válida     : {df_all["Pred_FT"].notna().sum()}')
print(f'  Com Pred_SC válida     : {df_all["Pred_SC"].notna().sum()}')
print()
print('Distribuição por categoria:')
cat_df = df_all.groupby('categoria').agg(
    total=('formula', 'count'),
    com_HSE=('gap_hse_true', lambda x: x.notna().sum()),
    UWBG_pred=('UWBG_FT', 'sum'),
).sort_values('total', ascending=False)
print(cat_df.to_string())


df_all.to_csv(OUTPUTS_DIR / 'all_materials_predictions.csv', index=False)

In [ ]:
from itables import show
import itables.options as opt

# Monta tabela para exibição
TABLE_COLS = {
    'formula'      : 'Fórmula',
    'categoria'    : 'Categoria',
    'ehull'        : 'ehull (eV/at)',
    'dyn_stab'     : 'Dyn_Stab',
    'gap_pbe'      : 'PBE (eV)',
    'gap_hse_true' : 'HSE_real (eV)',
    'Pred_FT'      : 'Pred_FT (eV)',
    'Pred_SC'      : 'Pred_SC (eV)',
    'UWBG_FT'      : 'UWBG_pred',
    'n_atoms'      : 'N_átomos',
}

df_show = (
    df_all
    .rename(columns=TABLE_COLS)
    [list(TABLE_COLS.values())]
    .sort_values('HSE_real (eV)', ascending=False, na_position='last')
    .reset_index(drop=True)
)

# Arredonda colunas numéricas
for col in ['ehull (eV/at)', 'PBE (eV)', 'HSE_real (eV)',
            'Pred_FT (eV)', 'Pred_SC (eV)', 'Pred_AL (eV)']:
    if col in df_show.columns:
        df_show[col] = df_show[col].round(4)

# Configuração itables
opt.maxBytes    = 0          # sem limite de bytes — mostra todos os dados
opt.lengthMenu  = [25, 50, 100, 250, 500]
opt.pageLength  = 50

show(
    df_show,
    caption='Materiais C2DB — predições comparadas  '
            '(Pred_FT = Fine-tune, Pred_SC = Scratch, Pred_AL = ALIGNN)',
    scrollX=True,
    scrollY='500px',
    classes='display nowrap compact',
    columnDefs=[
        {'className': 'dt-right', 'targets': list(range(2, 10))},
        {'width': '130px', 'targets': 0},
        {'width': '80px',  'targets': 1},
    ],
    buttons=['csv', 'excel'],
    layout={'top1': 'searchBuilder'},
)
